# Chapter 12: Which Planes Matter?

## Dependency density, effective work $D_l \times \theta_l$, and dependency-weighted bivectors

In previous chapters we built a rich geometric vocabulary: bivectors encode rotation planes, rotors compose across layers, holonomy measures curvature, and commutators quantify non-commutativity.  But a critical question remains: **which of these geometric features actually matter for the model's prediction?**

Not all layers contribute equally to the output.  The **dependency density** $D_l = \|\partial s / \partial H^{(l)}\|_F$ measures how sensitive the final prediction is to perturbations at layer $l$.  It is computed via a single backward pass through the network and gives a per-layer importance weight.

When we combine dependency with geometry, powerful insights emerge:

- **Effective work** $D_l \times \theta_l$ identifies layers that are both *important* (high $D_l$) and *geometrically active* (high rotation angle $\theta_l$).  A layer with a large rotation but zero dependency is doing irrelevant work.
- **Dependency-weighted bivectors** tell us which *planes* drive the prediction — not just which planes exist, but which planes the model actually relies on.
- **Dependency entropy** $H(D) = -\sum_l \hat{D}_l \log \hat{D}_l$ (where $\hat{D}_l = D_l / \sum D$) measures how *distributed* the computation is.  A low-entropy profile means the prediction hinges on a few layers; high entropy means many layers contribute — a signature of harder tasks.

**By the end of this chapter you will be able to:**
- Compute and visualize the dependency density profile $D_l$
- Identify the layers of highest effective work $D_l \times \theta_l$
- Extract the specific rotation planes that drive the prediction
- Use dependency entropy to classify task complexity

In [ ]:
import sys, os

# Ensure the project root is on the path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import ltg_ga

print("Imports OK")
print(f"NumPy {np.__version__}")

In [ ]:
# Load model and analyse with dependency computation enabled
model = ltg_ga.load_model("Qwen/Qwen2.5-7B")

prompt = "The capital of France is"
result = ltg_ga.analyse(prompt, model=model, compute_dependency=True)

print(f"Model: {model.name}")
print(f"Prompt: \"{prompt}\"")
print(f"Layers: {result.n_layers}, Tokens: {result.n_tokens}, Whitened dim: {result.k}")
print(f"Dependency computed: {result.dependency_profile is not None}")

## Dependency Profile

The **dependency density** at layer $l$ is:

$$D_l = \left\| \frac{\partial s}{\partial H^{(l)}} \right\|_F$$

where $s$ is the logit of the top predicted token at the final position.  This is computed via a single backward pass: the gradient of $s$ with respect to the hidden state at each layer gives $D_l$.

Three summary statistics help interpret the profile:

- **$D_{\mathrm{total}} = \sum_l D_l$**: Total gradient mass.  Larger values mean the prediction is more sensitive overall.
- **Dependency entropy $H(D) = -\sum_l \hat{D}_l \log \hat{D}_l$**: How spread out the dependency is across layers.  Maximum entropy $= \log L$ means perfectly uniform; low entropy means concentrated in a few layers.
- **Horizon-90**: The earliest layer $l^*$ such that $\sum_{l \leq l^*} \hat{D}_l \geq 0.90$.  Layers beyond the horizon carry 90% of the dependency — layers before it are relatively unimportant.

In [ ]:
# Plot dependency profile as bar chart with horizon-90 marked
D = result.dependency_profile
n_dep = len(D)

fig, ax = plt.subplots(figsize=(10, 4))

# Normalize for coloring
D_norm = D / (D.sum() + 1e-12)
colors = plt.cm.YlOrRd(D_norm / (D_norm.max() + 1e-12))

ax.bar(range(n_dep), D, color=colors, edgecolor='none', width=0.8)

# Mark horizon-90
if result.dep_horizon_90 is not None:
    ax.axvline(result.dep_horizon_90, color='#228833', linestyle='--',
               linewidth=2, label=f'Horizon-90 = layer {result.dep_horizon_90}')
    ax.legend(fontsize=10)

ax.set_xlabel('Layer')
ax.set_ylabel('Dependency density $D_l$')
ax.set_title(f'Dependency Profile — "{prompt}"')
ax.set_xlim(-0.5, n_dep - 0.5)
fig.tight_layout()

save_path = 'ch12_dependency.png'
fig.savefig(save_path, bbox_inches='tight', dpi=150)
print(f"Saved: {save_path}")

# Print summary statistics
print(f"\n=== Dependency Summary ===")
print(f"  D_total:        {result.dep_total:.4f}")
print(f"  Entropy:        {result.dep_entropy:.3f}  (max = {np.log(n_dep):.3f})")
print(f"  Horizon-90:     layer {result.dep_horizon_90} (of {n_dep})")
print(f"  Peak layer:     {D.argmax()} (D = {D.max():.4f})")
print(f"  Top-3 layers:   {np.argsort(D)[-3:][::-1].tolist()}")
plt.close(fig)

## Effective Work: Dependency $\times$ Rotation

Rotor angles $\theta_l$ tell us *how much* each layer rotates the hidden state.  Dependency $D_l$ tells us *how much each layer matters* for the prediction.  Neither quantity alone is sufficient:

- A layer with large $\theta_l$ but tiny $D_l$ is doing geometrically interesting but **functionally irrelevant** work.
- A layer with large $D_l$ but tiny $\theta_l$ is important but **geometrically passive** — it passes information through without transforming it much.

The **effective work** $W_l = D_l \times \theta_l$ identifies layers that are both important AND geometrically active.  These are the layers where the model is doing the real computational work that drives the prediction.

The two-panel visualization below shows how dependency and rotation relate across the network. The right panel highlights the top-3 effective-work layers — these are the layers you should focus on when interpreting the model.

In [ ]:
# Two-panel figure: dependency vs. rotation, and effective work
angles = result.angles
D = result.dependency_profile

# Align lengths: dependency has one entry per layer, angles has one per transition.
# Use the shorter length for alignment.
n_common = min(len(D), len(angles))
D_aligned = D[:n_common]
angles_aligned = angles[:n_common]
effective_work = D_aligned * angles_aligned

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Left panel: Overlay D_l (bars) and angles (line, twin axis) ---
layers = np.arange(n_common)
ax1.bar(layers, D_aligned, color='#BBCCEE', edgecolor='none', width=0.8,
        label='$D_l$ (dependency)', alpha=0.8)
ax1.set_xlabel('Layer')
ax1.set_ylabel('Dependency density $D_l$', color='#4477AA')
ax1.tick_params(axis='y', labelcolor='#4477AA')

ax1_twin = ax1.twinx()
ax1_twin.plot(layers, angles_aligned, color='#EE6677', linewidth=2,
              marker='o', markersize=3, label='$\\theta_l$ (angle)')
ax1_twin.set_ylabel('Rotation angle (rad)', color='#EE6677')
ax1_twin.tick_params(axis='y', labelcolor='#EE6677')

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_twin.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)
ax1.set_title('Dependency and Rotation Angle')

# --- Right panel: Effective work D_l * theta_l ---
bar_colors = np.full(n_common, '#BBCCEE', dtype=object)
top3_ew = np.argsort(effective_work)[-3:][::-1]
for idx in top3_ew:
    bar_colors[idx] = '#EE6677'

ax2.bar(layers, effective_work, color=bar_colors, edgecolor='none', width=0.8)

# Annotate top-3
for rank, idx in enumerate(top3_ew):
    ax2.annotate(f'#{rank+1}\n(L{idx})',
                 xy=(idx, effective_work[idx]),
                 ha='center', va='bottom', fontsize=8, fontweight='bold',
                 color='#CC3311')

ax2.set_xlabel('Layer')
ax2.set_ylabel('Effective work $D_l \\times \\theta_l$')
ax2.set_title('Effective Work (top-3 highlighted)')

fig.tight_layout()
save_path = 'ch12_effective_work.png'
fig.savefig(save_path, bbox_inches='tight', dpi=150)
print(f"Saved: {save_path}")

print(f"\nTop-3 effective-work layers:")
for rank, idx in enumerate(top3_ew):
    print(f"  #{rank+1}: Layer {idx}  |  D = {D_aligned[idx]:.4f}  |  "
          f"theta = {angles_aligned[idx]:.4f}  |  W = {effective_work[idx]:.4f}")
plt.close(fig)

## The Functionally Relevant Planes

For the top-3 effective-work layers, we can extract the **principal rotation plane** — the dominant plane in which that layer rotates the hidden state.  Combined with the dependency weight, this gives the most precise description available of what the model is actually doing:

> "At layer $l$, the model rotates the hidden state by $\theta_l$ radians in the plane spanned by $(v_1, v_2)$, and this rotation accounts for $X\%$ of the effective work."

This is a fundamentally different kind of interpretability from attention maps or feature visualizations.  It describes the model's computation in terms of **geometric transformations in representation space**, weighted by their actual impact on the output.

For each top layer, we also report:
- **$\kappa$** (condition number): how selective the metric deformation is
- **effective rank**: how many dimensions are actively stretched
- **principal plane angle and weight**: the dominant rotation plane

In [ ]:
# Identify top-3 layers by effective work and extract their principal planes
decompositions = result.rotor_field.decompositions
total_ew = effective_work.sum()

print(f"=== Functionally Relevant Planes ===")
print(f"Prompt: \"{prompt}\"")
print(f"Total effective work: {total_ew:.4f}\n")

for rank, idx in enumerate(top3_ew):
    print(f"--- #{rank+1}: Layer {idx} ---")
    print(f"  Dependency D_l:       {D_aligned[idx]:.4f}")
    print(f"  Rotation angle θ_l:   {angles_aligned[idx]:.4f} rad")
    print(f"  Effective work W_l:   {effective_work[idx]:.4f} "
          f"({effective_work[idx]/total_ew*100:.1f}% of total)")

    # Get the versor decomposition for this layer
    # decompositions are indexed by transition; match to the aligned index
    if idx < len(decompositions):
        vd = decompositions[idx]
        print(f"  Condition number κ:   {vd.condition_number:.2f}")
        print(f"  Effective rank:       {vd.effective_rank:.1f}")

        # Extract principal plane of the bivector
        planes = vd.bivector.principal_planes(n_planes=3)
        if planes:
            top_plane = planes[0]
            total_plane_weight = sum(p['weight'] for p in planes)
            print(f"  Principal plane:")
            print(f"    Angle (rotation in plane): {top_plane['angle']:.4f} rad")
            print(f"    Weight: {top_plane['weight']:.4f} "
                  f"({top_plane['weight']/total_plane_weight*100:.1f}% of bivector)")
            if len(planes) > 1:
                print(f"  2nd plane weight: {planes[1]['weight']:.4f} "
                      f"({planes[1]['weight']/total_plane_weight*100:.1f}%)")
    print()

## Dependency Entropy as Complexity

The **dependency entropy** provides a single number that characterizes the *complexity* of the model's computation for a given prompt:

$$H(D) = -\sum_l \hat{D}_l \log \hat{D}_l, \qquad \hat{D}_l = \frac{D_l}{\sum_j D_j}$$

- **Low entropy** (concentrated dependency): The prediction relies on a few layers. Typical for easy tasks like factual recall, where the answer is "stored" in a small part of the network.
- **High entropy** (distributed dependency): Many layers contribute to the prediction. Typical for hard tasks like multi-step reasoning, where the answer must be composed across many layers.

The maximum possible entropy is $\log L$ (perfectly uniform dependency across all $L$ layers).

Let us compare three prompts of increasing difficulty and observe how the dependency profile and entropy change.

In [ ]:
# Compare dependency profiles across prompts of different difficulty
difficulty_prompts = {
    "Easy (factual)":     "The capital of France is",
    "Medium (arithmetic)": "What is 347 plus 258? The answer is",
    "Hard (reasoning)":   "If all roses are flowers and some flowers fade quickly, then",
}

difficulty_results = {}
for label, prompt in difficulty_prompts.items():
    print(f"Analysing: {label} ...")
    difficulty_results[label] = ltg_ga.analyse(
        prompt, model=model, compute_dependency=True
    )

print("\nAll analyses complete.\n")

# Overlaid normalized dependency profiles
fig, ax = plt.subplots(figsize=(10, 5))
colors_diff = ['#4477AA', '#EE6677', '#228833']

for i, (label, r) in enumerate(difficulty_results.items()):
    D = r.dependency_profile
    D_norm = D / (D.sum() + 1e-12)
    ax.plot(D_norm, color=colors_diff[i], linewidth=1.8, alpha=0.85,
            label=f"{label}  (H = {r.dep_entropy:.2f})")

ax.set_xlabel('Layer')
ax.set_ylabel('Normalized dependency $\\hat{D}_l$')
ax.set_title('Dependency Profiles: Easy vs. Medium vs. Hard')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
fig.tight_layout()

save_path = 'ch12_dependency_entropy.png'
fig.savefig(save_path, bbox_inches='tight', dpi=150)
print(f"Saved: {save_path}")

# Summary table
max_entropy = np.log(result.n_layers)
print(f"\n{'Task':<25} {'Entropy':>8} {'H/H_max':>8} {'Horizon-90':>11} {'D_total':>8}")
print("-" * 65)
for label, r in difficulty_results.items():
    h_ratio = r.dep_entropy / max_entropy if max_entropy > 0 else 0
    print(f"{label:<25} {r.dep_entropy:>8.3f} {h_ratio:>8.3f} "
          f"{r.dep_horizon_90:>11d} {r.dep_total:>8.4f}")

print(f"\nMaximum possible entropy: {max_entropy:.3f} (= log {result.n_layers})")
plt.close(fig)

## Exercises

1. **Effective work for all layers.** Extend the top-3 analysis: for *every* layer, compute $W_l = D_l \times \theta_l$ and extract its principal plane. Create a scatter plot with $D_l$ on the x-axis and $\theta_l$ on the y-axis, with point size proportional to $W_l$. Which quadrant do most layers fall into?

2. **Dependency-weighted capacity.** Recall from Chapter 11 that $C_{\mathrm{eff}}$ weights commutator norms by $\sqrt{D_i D_j}$.  Compare $C_{\mathrm{eff}}$ to $C_{\mathrm{acc}}$ for the three difficulty prompts above.  Does the ratio $C_{\mathrm{eff}} / C_{\mathrm{acc}}$ increase with task difficulty?

3. **Plane overlap across top layers.** For the top-3 effective-work layers, extract their principal planes and compute pairwise **plane similarity** (as the absolute cosine of the angle between the plane bivectors).  Are the top layers rotating in the same plane, or in orthogonal planes?

4. **Horizon shift.** The horizon-90 layer marks where 90% of dependency is accumulated.  Compare the horizon-90 across 5+ prompts of varying complexity.  Is there a consistent relationship between horizon-90 and dependency entropy?

5. **Pruning simulation.** Zero out the dependency of all layers *below* the horizon-90 (i.e., set $D_l = 0$ for $l < l^*$).  Recompute $C_{\mathrm{eff}}$ with this pruned dependency profile.  How much effective capacity is lost?  This simulates what would happen if you skipped the early layers.

6. **Dependency under different models.** If you have access to multiple model sizes, compare their dependency entropy on the same prompt.  Do larger models distribute computation more evenly, or do they concentrate it more?